In [ ]:
import os
from pathlib import Path
# 現在の作業ディレクトリを確認し、'scripts' にいる場合は親ディレクトリに移動
current_dir = Path.cwd()
if current_dir.name == "scripts":
    os.chdir(current_dir.parent)

In [ ]:
# Fear Conditioning

from pathlib import Path  # noqa: F811

import numpy as np  # noqa: F811
import pandas as pd

# 自動でエクセルファイルを読み込む

folder = r"FC"

files = sorted(Path(folder).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート

if not files:
    print(f"{folder} の中に .xls ファイルはありません")
else:
    dfs = [] # 空のリストを作成して、各ファイルのデータフレームを格納
    for file in files: # ファイルごとにループ
        df = (
        pd.read_excel(file) 
        .iloc[2:8, [4]]  # 2行目から7行目まで、4列目を抽出
        .assign(Freezing = lambda df: df['Interval.3'] / 60 * 100 ) # Freezing Time (%) を計算し列に追加
        [['Freezing']] # Freezing列のみを残す
        .assign(
        Time  = lambda df: list(range(1, len(df) + 1)), # Time列に1から行数までの連番を追加
        No = lambda df: Path(file).stem,  # No列にpathからファイル名を抽出して追加
        Group = lambda df: np.select( # Group列に条件に応じた値を追加
            condlist=[ # 条件のリスト：No列に各文字列が含まれているか
                df['No'].str.contains('SED'),
                df['No'].str.contains('LIE'),
                df['No'].str.contains('MOE')
            ],
            choicelist=['SED', 'LIE', 'MOE'], # 条件にマッチしたときに入れる値のリスト
            default='Other' # どれにも当てはまらない場合のデフォルト値
            )
        )
     )
        dfs.append(df) # データフレームをリストに追加

    dataFC = pd.concat(dfs, ignore_index=True) # リスト内のデータフレームを縦に結合して1つのデータフレームにする

    print(f"{len(files)} 件の .xls ファイルを読み込みました") # 読み込んだファイル数を表示
    print(dataFC) # データフレームの内容を表示

24 件の .xls ファイルを読み込みました
      Freezing  Time        No Group
0          0.0     1  100FCMOE   MOE
1          0.0     2  100FCMOE   MOE
2          0.0     3  100FCMOE   MOE
3         47.6     4  100FCMOE   MOE
4         43.5     5  100FCMOE   MOE
..         ...   ...       ...   ...
139        0.0     2   99FCMOE   MOE
140        0.0     3   99FCMOE   MOE
141  14.566667     4   99FCMOE   MOE
142       60.8     5   99FCMOE   MOE
143  71.966667     6   99FCMOE   MOE

[144 rows x 4 columns]


In [ ]:
# Fear Extinction

from pandas.core.arrays import categorical
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# 旧形式 .xls の OLE2 警告を抑制
# warnings.filterwarnings("ignore", message=".*OLE2 inconsistency.*")

def infer_group(name: str) -> str: # ファイル名からグループを推測する関数
    if "SED" in name:
        return "SED"
    elif "LIE" in name:
        return "LIE"
    elif "MOE" in name:
        return "MOE"
    return "Other"

def read_extinction_per3(files):   # 3分ごとのデータを読み込む関数
    rows = [] # 空のリストを作成して、各ファイルのデータを格納

    for file in files:
        # 旧 .xls では pandas で警告が出ることがあるので抑制
        # with warnings.catch_warnings():
            # warnings.filterwarnings("ignore", message=".*OLE2 inconsistency.*")
        df = pd.read_excel(file)

        col5 = pd.to_numeric(df.iloc[:, 4], errors="coerce")  # 5列目（E列）
        bins = [ # 3分毎のbinの範囲とラベル
            (3, 8, "3"),
            (9, 14, "6"),
            (15, 20, "9"),
            (21, 26, "12"),
            (27, 32, "15"),
        ]

        for start, end, time_label in bins: # 3分毎のbinごとにデータを処理
            freezing = (
                col5.iloc[start - 1:end] # 3:8, 9:14, ...
                .astype(float)           # 数値に変換
                .mul(100 / 30)           # 30秒ごとのデータを3分ごとの平均に変換
                .mean()                  # 平均値を計算  
            )
            rows.append({
                "No": Path(file).stem,   # ファイル名を追加
                "Time": time_label,      # ラベルを追加
                "Freezing": freezing,    # 平均値を追加
                "Group": infer_group(Path(file).stem),  # グループを推測して追加
            })

    return pd.DataFrame(rows)

# 既存のフォルダ
folder_Ex1 = r"Ex1"
folder_Ex2 = r"Ex2"

files_Ex1 = sorted(Path(folder_Ex1).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート
files_Ex2 = sorted(Path(folder_Ex2).glob("*.xls")) # フォルダ内の .xls ファイルを取得してソート

# 3分ごとのデータ
if not files_Ex1 and not files_Ex2:
    print("指定されたフォルダに .xls ファイルはありません")
else:
    dataEx1_per3 = read_extinction_per3(files_Ex1) # Ex1の3分ごとのデータを読み込む
    dataEx2_per3 = read_extinction_per3(files_Ex2) # Ex2の3分ごとのデータを読み込む

    print(f"Ex1: {len(files_Ex1)} 件, Ex2: {len(files_Ex2)} 件") 
    print(dataEx1_per3.head()) 
    print(dataEx2_per3.head())

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but 

In [ ]:
# anovakun用にwide形式のデータセットに整形
group_order = ["SED", "LIE", "MOE"] # Group列の並べ替え
dataFC["Group"] = pd.Categorical( 
    dataFC["Group"],
    categories=group_order,
    ordered=True
)

wide_FC = (
    dataFC
    .pivot_table(
        index=["No", "Group"], 
        columns="Time",
        values="Freezing",
        aggfunc="mean"
    )
    .reset_index()
)

print(wide_FC)

Time        No Group          1          2          3          4          5  \
0     100FCMOE   MOE        0.0        0.0        0.0       47.6       43.5   
1     101FCSED   SED        0.0        0.0        0.0       30.2  50.466667   
2     102FCSED   SED        0.0        0.0   5.566667  62.133333       57.6   
3     103FCSED   SED   3.366667        0.0        0.0       21.2  43.333333   
4     104FCSED   SED        0.0        0.0        0.0        4.1       28.5   
5     105FCLIE   LIE        0.0        0.0  54.266667        0.0   4.066667   
6     106FCLIE   LIE        0.0        0.0       26.3  20.433333  55.366667   
7     107FCLIE   LIE        0.0        0.0        0.0  22.066667       26.0   
8     108FCLIE   LIE        0.0        0.0        0.0       31.8  58.433333   
9     109FCMOE   MOE  21.833333   5.033333  13.266667       16.2       45.0   
10    110FCMOE   MOE        0.0        4.5  25.866667       44.7       65.5   
11    111FCMOE   MOE       15.8        0.0        0.

In [19]:
# anovakun用 Fear Extinction day 1

group_order = ["SED", "LIE", "MOE"]
dataEx1_per3["Group"] = pd.Categorical(
    dataEx1_per3["Group"],
    categories=group_order,
    ordered=True
)

wide_Ex1 = (
    dataEx1_per3
    .pivot_table(
        index=["No", "Group"], 
        columns="Time",
        values="Freezing",
        aggfunc="mean"
    )
    .reindex(columns=['3','6','9','12','15'])
    .reset_index()
)

wide_Ex1["Group"] = pd.Categorical(
    wide_Ex1["Group"],
    categories=["SED","LIE","MOE"],
    ordered=True
)

print(wide_Ex1)

Time         No Group           3           6           9         12  \
0     100MOEEx1   MOE   83.444444   91.944444   71.200000  37.100000   
1     101SEDEx1   SED   83.566667   82.777778   42.155556  45.822222   
2     102SEDEx1   SED   90.500000   96.300000   95.177778  89.411111   
3     103SEDEx1   SED   98.933333  100.000000   99.800000  95.144444   
4     104SEDEx1   SED   93.988889   90.033333   90.733333  91.733333   
5     105LIEEx1   LIE   86.722222   65.444444   61.622222  24.233333   
6     106LIEEx1   LIE   86.655556   73.344444   67.477778  16.255556   
7     107LIEEx1   LIE   74.622222   43.333333   20.733333  33.077778   
8     108LIEEx1   LIE   91.688889   75.000000   70.011111  40.911111   
9     109MOEEx1   MOE   93.033333   97.088889   30.133333  34.166667   
10    110MOEEx1   MOE   98.966667   97.222222   64.388889  44.388889   
11    111MOEEx1   MOE   99.200000   61.766667   34.200000  34.422222   
12    112MOEEx1   MOE   69.833333   99.811111   77.100000  50.84

In [ ]:
# anovakunをR用に改編し、統計検定 Fear Conditioning
import pandas as pd
from stats import anovakun

anova_table_FC = anovakun(
    dataset=wide_FC,
    design="AsB",
    group_col="Group",
    time_cols=list(range(1,7)),
    hf=True,
    peta=True
)

print(anova_table_FC)

Source                  SS      df          MS         F         p   p.eta2   GGeps     p(GG)   HFeps     p(HF)
Group(A)           113.543    2.00      56.771     0.150    0.8619   0.0141                                    
S/A(error)        7963.869   21.00     379.232                                                                 
Time(B)          84911.381    5.00   16982.276    88.604    0.0000   0.8084   0.680    0.0000   0.905    0.0000
A x B             2943.335   10.00     294.334     1.536    0.1370   0.1276   0.680    0.1711   0.905    0.1462
BxS/A(error)     20124.885  105.00     191.666                                                                 


In [22]:
# Fear Extinction

anova_table_Ex1 = anovakun(
    dataset=wide_Ex1,
    design="AsB",
    group_col="Group",
    time_cols=["3","6","9","12","15"],
    hf=True,
    peta=True
)

print(anova_table_Ex1)

Source                  SS      df          MS         F         p   p.eta2   GGeps     p(GG)   HFeps     p(HF)
Group(A)         18192.466    2.00    9096.233    12.123    0.0003   0.5359                                    
S/A(error)       15757.354   21.00     750.350                                                                 
Time(B)          29755.985    4.00    7438.996    37.025    0.0000   0.6381   0.597    0.0000   0.743    0.0000
A x B             7601.804    8.00     950.225     4.729    0.0001   0.3105   0.597    0.0015   0.743    0.0005
BxS/A(error)     16877.230   84.00     200.919                                                                 
